<a href="https://colab.research.google.com/github/Wasey23/CI-345-Machine-Learning-Project/blob/main/Support_Machine_Vector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Support Vector Machine (SVM) Model for Movie Success Prediction

## Project Overview

This notebook develops a Support Vector Machine (SVM) model to predict whether a movie is commercially successful.

A movie is classified as:

- 1 = Hit Movie
- 0 = Flop Movie

where:

revenue > 2 × budget

## Import Libraries

In [4]:
import pandas as pd
import numpy as np

from google.colab import drive

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.preprocessing import (
    StandardScaler,
    MultiLabelBinarizer
)

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

## Load Dataset

In [5]:
tmbd_processed = pd.read_csv(
    "tmdb_movies_data.csv"
)

tmbd_processed.head()

,id,imdb_id,popularity,budget,revenue,original_title,cast,homepage,director,tagline,...,overview,runtime,genres,production_companies,release_date,vote_count,vote_average,release_year,budget_adj,revenue_adj
0,135397,tt0369610,32.985763,150000000,1513528810,Jurassic World,Chris Pratt|Bryce Dallas Howard|Irrfan Khan|Vi...,http://www.jurassicworld.com/,Colin Trevorrow,The park is open.,...,Twenty-two years after the events of Jurassic ...,124,Action|Adventure|Science Fiction|Thriller,Universal Studios|Amblin Entertainment|Legenda...,6/9/2015,5562,6.5,2015,137999939.3,1.392446e+09
1,76341,tt1392190,28.419936,150000000,378436354,Mad Max: Fury Road,Tom Hardy|Charlize Theron|Hugh Keays-Byrne|Nic...,http://www.madmaxmovie.com/,George Miller,What a Lovely Day.,...,An apocalyptic story set in the furthest reach...,120,Action|Adventure|Science Fiction|Thriller,Village Roadshow Pictures|Kennedy Miller Produ...,5/13/2015,6185,7.1,2015,137999939.3,3.481613e+08
2,262500,tt2908446,13.112507,110000000,295238201,Insurgent,Shailene Woodley|Theo James|Kate Winslet|Ansel...,http://www.thedivergentseries.movie/#insurgent,Robert Schwentke,One Choice Can Destroy You,...,Beatrice Prior must confront her inner demons ...,119,Adventure|Science Fiction|Thriller,Summit Entertainment|Mandeville Films|Red Wago...,3/18/2015,2480,6.3,2015,101199955.5,2.716190e+08
3,140607,tt2488496,11.173104,200000000,2068178225,Star Wars: The Force Awakens,Harrison Ford|Mark Hamill|Carrie Fisher|Adam D...,http://www.starwars.com/films/star-wars-episod...,J.J. Abrams,Every generation has a story.,...,Thirty years after defeating the Galactic Empi...,136,Action|Adventure|Science Fiction|Fantasy,Lucasfilm|Truenorth Productions|Bad Robot,12/15/2015,5292,7.5,2015,183999919.0,1.902723e+09
4,168259,tt2820852,9.335014,190000000,1506249360,Furious 7,Vin Diesel|Paul Walker|Jason Statham|Michelle ...,http://www.furious7.com/,James Wan,Vengeance Hits Home,...,Deckard Shaw seeks revenge against Dominic Tor...,137,Action|Crime|Thriller,Universal Pictures|Original Film|Media Rights ...,4/1/2015,2947,7.3,2015,174799923.1,1.385749e+09


In [ ]:
tmbd_processed.drop_duplicates(inplace=True)
tmbd_processed.duplicated().sum()

np.int64(0)

## Feature Engineering

In [ ]:
tmbd_processed['profit'] = (
    tmbd_processed['revenue']
    - tmbd_processed['budget']
)

tmbd_processed['roi'] = (
    tmbd_processed['revenue']
    / tmbd_processed['budget']
)

## Encode Genres

In [ ]:
genres_encoded = (
    tmbd_processed['genres']
    .fillna('') # Fill NaN values with empty strings
    .str.split('|')
)

# Ensure that empty strings (from original NaN or empty genre entries) are not treated as genres
genres_encoded = genres_encoded.apply(lambda x: [genre for genre in x if genre])

mlb = MultiLabelBinarizer()

genre_df = pd.DataFrame(
    mlb.fit_transform(genres_encoded),
    columns=mlb.classes_
)

tmbd_processed = pd.concat(
    [tmbd_processed, genre_df],
    axis=1
)

## Feature Selection

In [ ]:
tmbd_processed['is_hit'] = (tmbd_processed['revenue'] > 2 * tmbd_processed['budget']).astype(int)

# Convert 'release_date' to datetime and extract the month
tmbd_processed['release_month'] = pd.to_datetime(tmbd_processed['release_date']).dt.month

features = [
    'popularity',
    'runtime',
    'vote_count',
    'vote_average',
    'budget_adj',
    'release_month'
]

features += list(mlb.classes_)

X = tmbd_processed[features]

y = tmbd_processed['is_hit']

## Class Distribution

In [ ]:
print(y.value_counts())

is_hit
0    7850
1    3016
Name: count, dtype: int64


## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

## Baseline SVM Model

In [ ]:
from sklearn.impute import SimpleImputer

# Impute missing values in scaled data
imputer = SimpleImputer(strategy='mean')
X_train_scaled = imputer.fit_transform(X_train_scaled)
X_test_scaled = imputer.transform(X_test_scaled)

baseline_model = SVC(
    kernel='rbf',
    random_state=42
)

baseline_model.fit(
    X_train_scaled,
    y_train
)

baseline_pred = baseline_model.predict(
    X_test_scaled
)

## Baseline Model Performance

In [ ]:
baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

baseline_precision = precision_score(
    y_test,
    baseline_pred
)

baseline_recall = recall_score(
    y_test,
    baseline_pred
)

baseline_f1 = f1_score(
    y_test,
    baseline_pred
)

print(
    classification_report(
        y_test,
        baseline_pred
    )
)

              precision    recall  f1-score   support

           0       0.79      0.96      0.86      1571
           1       0.74      0.33      0.45       603

    accuracy                           0.78      2174
   macro avg       0.77      0.64      0.66      2174
weighted avg       0.78      0.78      0.75      2174



## Hyperparameter Tuning

In [ ]:
param_grid = {
    'C': [1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 0.1]
}

grid = GridSearchCV(
    SVC(),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(
    X_train_scaled,
    y_train
)

print(
    "Best Parameters:",
    grid.best_params_
)

print(
    "Best Score:",
    grid.best_score_
)

Best Parameters: {'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
Best Score: 0.7838244651364672


## Tuned SVM Model

In [ ]:
best_model = grid.best_estimator_

best_model.fit(
    X_train_scaled,
    y_train
)

tuned_pred = best_model.predict(
    X_test_scaled
)

## Tuned Model Performance

In [ ]:
tuned_accuracy = accuracy_score(
    y_test,
    tuned_pred
)

tuned_precision = precision_score(
    y_test,
    tuned_pred
)

tuned_recall = recall_score(
    y_test,
    tuned_pred
)

tuned_f1 = f1_score(
    y_test,
    tuned_pred
)

print(
    classification_report(
        y_test,
        tuned_pred
    )
)

              precision    recall  f1-score   support

           0       0.78      0.96      0.86      1571
           1       0.75      0.29      0.42       603

    accuracy                           0.78      2174
   macro avg       0.76      0.63      0.64      2174
weighted avg       0.77      0.78      0.74      2174



## Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    tuned_pred
)

print(cm)

[[1513   58]
 [ 429  174]]


## Cross Validation

In [ ]:
scores = cross_val_score(
    best_model,
    X_train_scaled,
    y_train,
    cv=5
)

print(scores)

print(
    "Average Score:",
    scores.mean()
)

[0.77515814 0.7826337  0.79804373 0.78365938 0.783084  ]
Average Score: 0.7845157892020267


## Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': [
        'Baseline SVM',
        'Tuned SVM'
    ],

    'Accuracy': [
        baseline_accuracy,
        tuned_accuracy
    ],

    'Precision': [
        baseline_precision,
        tuned_precision
    ],

    'Recall': [
        baseline_recall,
        tuned_recall
    ],

    'F1 Score': [
        baseline_f1,
        tuned_f1
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,Baseline SVM,0.781969,0.743396,0.326700,0.453917
1,Tuned SVM,0.775989,0.750000,0.288557,0.416766


## Conclusion

The Support Vector Machine model was trained to predict movie success.

Hyperparameter tuning was performed using GridSearchCV to improve model performance. The tuned model was evaluated using accuracy, precision, recall, and F1-score and compared against the baseline SVM model.

Cross-validation was also conducted to assess the model's ability to generalize to unseen data.